# Оценка машинного перевода с французского на английский

**Задача:** построить воспроизводимый пайплайн инференса MarianMT и сравнить переводы с эталонами по нескольким метрикам.

**Стек:** Transformers, Hugging Face Datasets/Evaluate, PyTorch, pandas.

Метрики: SacreBLEU, TER, chrF++ и BERTScore. Метрика AER не используется: она требует эталонного выравнивания слов, которого в выбранном датасете нет.


In [ ]:
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
import evaluate
from transformers import MarianMTModel, MarianTokenizer

MODEL_NAME = "Helsinki-NLP/opus-mt-fr-en"
DATASET_NAME = "Fannyjrd/FairTranslate_fr"
SAMPLE_SIZE = 100
BATCH_SIZE = 16
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
dataset = load_dataset(DATASET_NAME, split=f"train[:{SAMPLE_SIZE}]")
required_columns = {"french", "english"}
missing = required_columns - set(dataset.column_names)
if missing:
    raise ValueError(f"В датасете отсутствуют колонки: {sorted(missing)}")

tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)
model = MarianMTModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

sources = [str(item["french"]) for item in dataset]
references = [str(item["english"]) for item in dataset]
print(f"Примеров: {len(sources)}; устройство: {DEVICE}")


In [ ]:
def translate_batch(texts: list[str]) -> list[str]:
    encoded = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256,
    ).to(DEVICE)
    with torch.inference_mode():
        generated = model.generate(
            **encoded,
            num_beams=4,
            max_new_tokens=256,
        )
    return tokenizer.batch_decode(generated, skip_special_tokens=True)


predictions = []
for start in range(0, len(sources), BATCH_SIZE):
    predictions.extend(translate_batch(sources[start : start + BATCH_SIZE]))

assert len(predictions) == len(references)


In [ ]:
sacrebleu = evaluate.load("sacrebleu")
ter = evaluate.load("ter")
chrf = evaluate.load("chrf")
bertscore = evaluate.load("bertscore")

bert = bertscore.compute(
    predictions=predictions,
    references=references,
    lang="en",
)

metrics = pd.Series(
    {
        "SacreBLEU ↑": sacrebleu.compute(
            predictions=predictions,
            references=[[text] for text in references],
        )["score"],
        "TER ↓": ter.compute(
            predictions=predictions,
            references=references,
        )["score"],
        "chrF++ ↑": chrf.compute(
            predictions=predictions,
            references=references,
            word_order=2,
        )["score"],
        "BERTScore F1 ↑": float(np.mean(bert["f1"])),
    },
    name="value",
)
metrics.round(4).to_frame()


In [ ]:
examples = pd.DataFrame(
    {
        "source_fr": sources,
        "reference_en": references,
        "prediction_en": predictions,
    }
)
pd.set_option("display.max_colwidth", 120)
examples.head(10)


## Выводы и ограничения

Несколько метрик нужны потому, что они оценивают разные свойства перевода. SacreBLEU чувствителен к совпадению слов и n-грамм, chrF++ лучше учитывает морфологические варианты, TER измеряет объём правок, а BERTScore — семантическое сходство. Для надёжного сравнения моделей следует увеличить тестовую выборку и добавить ручную оценку беглости и адекватности.
